# CS:GO Raunt Sonucu Tahmini

## Kütüphaneler ve Veri Yükleme

In [8]:
import zipfile
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix

with zipfile.ZipFile('csdata.zip', 'r') as z:
    with z.open('mm_master_demos.csv') as f:
        df = pd.read_csv(f)

## Veri Hazırlama ve Özellik Mühendisliği

In [9]:
# Oyuncu/hasar bazlı tekrar eden satırları raunt bazına indirge
df_temiz = df[['file', 'round', 'map', 'winner_side', 'ct_eq_val', 't_eq_val']]
df_rauntlar = df_temiz.drop_duplicates(subset=['file', 'round']).copy()

# 1-15 arası Takım_A T'dedir, 16 ve sonrasında Takım_B T'ye geçer
df_rauntlar['t_team'] = np.where(df_rauntlar['round'] <= 15, 'Takim_A', 'Takim_B')
df_rauntlar['ct_team'] = np.where(df_rauntlar['round'] <= 15, 'Takim_B', 'Takim_A')

df_rauntlar['kazanan_takim'] = np.where(df_rauntlar['winner_side'] == 'Terrorist', df_rauntlar['t_team'], df_rauntlar['ct_team'])

In [10]:
df_rauntlar['a_win'] = (df_rauntlar['kazanan_takim'] == 'Takim_A').astype(int)
df_rauntlar['b_win'] = (df_rauntlar['kazanan_takim'] == 'Takim_B').astype(int)

# Takım A'nın mevcut raunttaki ekipman değeri farkı (Ekonomi avantajı)
df_rauntlar['a_eq_diff'] = np.where(
    df_rauntlar['round'] <= 15,
    df_rauntlar['t_eq_val'] - df_rauntlar['ct_eq_val'],
    df_rauntlar['ct_eq_val'] - df_rauntlar['t_eq_val']
)

df_rauntlar['a_is_t'] = np.where(df_rauntlar['t_team'] == 'Takim_A', 1, 0)
df_rauntlar['map_side'] = df_rauntlar['map'] + '_' + np.where(df_rauntlar['a_is_t'] == 1, 'T', 'CT')

In [11]:
def seri_hesapla(seri):
    sayac = 0
    lose_streak = []
    for i in seri:
        if i == 0:
            sayac += 1
        else:
            sayac = 0
        lose_streak.append(sayac)
    return lose_streak

df_rauntlar['a_losing_streak'] = df_rauntlar.groupby('file')['a_win'].transform(seri_hesapla)
df_rauntlar['b_losing_streak'] = df_rauntlar.groupby('file')['b_win'].transform(seri_hesapla)

# Veri sızıntısını önlemek için mevcut raundun değil, önceki raundun serisini al
df_rauntlar['a_prev_streak'] = df_rauntlar.groupby('file')['a_losing_streak'].shift(1).fillna(0)
df_rauntlar['b_prev_streak'] = df_rauntlar.groupby('file')['b_losing_streak'].shift(1).fillna(0)

## Model Eğitimi ve Değerlendirme

In [12]:
model_sutunlar = ['a_eq_diff', 'a_prev_streak', 'b_prev_streak', 'map_side', 'a_win']
df_model = df_rauntlar[model_sutunlar].dropna().copy()
df_model = pd.get_dummies(df_model, drop_first=True)

X = df_model.drop(columns=['a_win'])
y = df_model['a_win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10)

# Test setinin istatistikleri eğitime karışmasın diye scaler sadece train verisinden öğrenir
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train, y_train);

In [13]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

basari_skoru = accuracy_score(y_test, y_pred)
roc_skoru = roc_auc_score(y_test, y_prob)

print(f"Modelin Doğruluk Skoru : %{basari_skoru * 100:.2f}")
print(f"ROC-AUC Skoru          : {roc_skoru:.4f}")

Modelin Doğruluk Skoru : %65.96
ROC-AUC Skoru          : 0.7191
